In [ ]:
import os
import time
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from neo4j import GraphDatabase
from torch_geometric.data import Data
from torch_geometric.datasets import Planetoid
import torch_geometric.transforms as T
from torch_geometric.nn import (
    MessagePassing,
    GCNConv,
    SAGEConv,
    GATConv,
    GINConv
)

# ==============================================================================
# 1. NEO4J CREDENTIALS & CONFIGURATION
# ==============================================================================
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")


# ==============================================================================
# 2. NEO4J INGESTION & EXTRACTION PIPELINE
# ==============================================================================
def populate_cora_in_neo4j(driver, data):
    """
    Uploads the Cora dataset into Neo4j as (:Paper) nodes and [:CITES] edges.
    """
    print("📥 Ingesting Cora dataset into Neo4j...")
    with driver.session() as session:
        # Clear existing Paper nodes
        session.run("MATCH (p:Paper) DETACH DELETE p")

        # 1. Ingest Nodes in batches
        node_records = []
        for idx in range(data.num_nodes):
            node_records.append({
                "id": int(idx),
                "features": data.x[idx].tolist(),
                "label": int(data.y[idx].item()),
                "train_mask": bool(data.train_mask[idx].item()),
                "val_mask": bool(data.val_mask[idx].item()),
                "test_mask": bool(data.test_mask[idx].item())
            })

        batch_size = 500
        for i in range(0, len(node_records), batch_size):
            batch = node_records[i:i + batch_size]
            session.run("""
            UNWIND $batch AS row
            CREATE (p:Paper {
                id: row.id,
                features: row.features,
                label: row.label,
                train_mask: row.train_mask,
                val_mask: row.val_mask,
                test_mask: row.test_mask
            })
            """, batch=batch)

        # 2. Ingest Edges in batches
        edge_records = []
        edges_src = data.edge_index[0].tolist()
        edges_dst = data.edge_index[1].tolist()
        for src, dst in zip(edges_src, edges_dst):
            edge_records.append({"src": int(src), "dst": int(dst)})

        for i in range(0, len(edge_records), batch_size):
            batch = edge_records[i:i + batch_size]
            session.run("""
            UNWIND $batch AS row
            MATCH (src:Paper {id: row.src})
            MATCH (dst:Paper {id: row.dst})
            CREATE (src)-[:CITES]->(dst)
            """, batch=batch)

    print("✅ Cora graph populated in Neo4j.\n")


def load_cora_from_neo4j(driver) -> Data:
    """
    Extracts node features, masks, labels, and adjacency from Neo4j into a PyG Data object.
    """
    print("🔌 Extracting Graph from Neo4j...")
    with driver.session() as session:
        # 1. Fetch all nodes
        nodes_result = session.run("""
        MATCH (p:Paper)
        RETURN p.id AS id, p.features AS features, p.label AS label,
               p.train_mask AS train_mask, p.val_mask AS val_mask, p.test_mask AS test_mask
        ORDER BY p.id ASC
        """)

        features, labels, train_mask, val_mask, test_mask = [], [], [], [], []
        id_to_index = {}

        for idx, record in enumerate(nodes_result):
            node_id = record["id"]
            id_to_index[node_id] = idx
            features.append(record["features"])
            labels.append(record["label"])
            train_mask.append(record["train_mask"])
            val_mask.append(record["val_mask"])
            test_mask.append(record["test_mask"])

        # 2. Fetch all citation relationships
        edges_result = session.run("""
        MATCH (src:Paper)-[:CITES]->(dst:Paper)
        RETURN src.id AS src, dst.id AS dst
        """)

        src_indices, dst_indices = [], []
        for record in edges_result:
            src_indices.append(id_to_index[record["src"]])
            dst_indices.append(id_to_index[record["dst"]])

    x = torch.tensor(features, dtype=torch.float)
    y = torch.tensor(labels, dtype=torch.long)
    edge_index = torch.tensor([src_indices, dst_indices], dtype=torch.long)
    train_mask = torch.tensor(train_mask, dtype=torch.bool)
    val_mask = torch.tensor(val_mask, dtype=torch.bool)
    test_mask = torch.tensor(test_mask, dtype=torch.bool)

    pyg_data = Data(
        x=x,
        edge_index=edge_index,
        y=y,
        train_mask=train_mask,
        val_mask=val_mask,
        test_mask=test_mask
    )

    print(f"✅ Loaded PyG Data: {pyg_data.num_nodes} nodes, {pyg_data.num_edges} edges, {pyg_data.num_node_features} features.\n")
    return pyg_data


# ==============================================================================
# 3. BASE MESSAGE PASSING (VANILLA GNN)
# ==============================================================================
class VanillaGNNConv(MessagePassing):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__(aggr='add')  # Sum message aggregator
        self.lin = nn.Linear(in_channels, out_channels)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        return self.propagate(edge_index, x=x)

    def message(self, x_j: torch.Tensor) -> torch.Tensor:
        return x_j

    def update(self, aggr_out: torch.Tensor) -> torch.Tensor:
        return self.lin(aggr_out)


# ==============================================================================
# 4. UNIFIED BENCHMARK GNN WRAPPER
# ==============================================================================
class BenchmarkGNN(nn.Module):
    def __init__(self, model_type: str, in_channels: int, hidden_channels: int, out_channels: int, heads: int = 8, dropout: float = 0.5):
        super().__init__()
        self.model_type = model_type
        self.dropout = dropout

        if model_type == 'Vanilla GNN (Graph Neural Network)':
            self.conv1 = VanillaGNNConv(in_channels, hidden_channels)
            self.conv2 = VanillaGNNConv(hidden_channels, out_channels)
        elif model_type == 'GCN (Graph Convolutional Network)':
            self.conv1 = GCNConv(in_channels, hidden_channels)
            self.conv2 = GCNConv(hidden_channels, out_channels)
        elif model_type == 'GraphSAGE (Sample and Aggregate)':
            self.conv1 = SAGEConv(in_channels, hidden_channels, aggr='mean')
            self.conv2 = SAGEConv(hidden_channels, out_channels, aggr='mean')
        elif model_type == 'GAT (Graph Attention Network)':
            self.conv1 = GATConv(in_channels, hidden_channels, heads=heads, dropout=dropout)
            self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=1, concat=False, dropout=dropout)
        elif model_type == 'GIN (Graph Isomorphism Network)':
            mlp1 = nn.Sequential(
                nn.Linear(in_channels, hidden_channels),
                nn.ReLU(),
                nn.Linear(hidden_channels, hidden_channels)
            )
            mlp2 = nn.Sequential(
                nn.Linear(hidden_channels, hidden_channels),
                nn.ReLU(),
                nn.Linear(hidden_channels, out_channels)
            )
            self.conv1 = GINConv(mlp1, train_eps=True)
            self.conv2 = GINConv(mlp2, train_eps=True)
        else:
            raise ValueError(f"Unknown model architecture: {model_type}")

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)


# ==============================================================================
# 5. TRAINING & EVALUATION FUNCTIONS
# ==============================================================================
def train_step(model, data, optimizer, criterion):
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()


@torch.no_grad()
def evaluate_splits(model, data):
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=-1)

    accuracies = []
    for mask in [data.train_mask, data.val_mask, data.test_mask]:
        correct = pred[mask].eq(data.y[mask]).sum().item()
        accuracies.append(correct / mask.sum().item())
    return accuracies  # [train_acc, val_acc, test_acc]


# ==============================================================================
# 6. BENCHMARK EXECUTION PIPELINE
# ==============================================================================
def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Executing PyTorch Geometric Benchmark on: {device}\n")

    # Connect to Neo4j
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

    try:
        # Load local Cora dataset and sync to Neo4j database once
        raw_cora = Planetoid(root='/tmp/Cora', name='Cora', transform=T.NormalizeFeatures())
        populate_cora_in_neo4j(driver, raw_cora[0])

        # Load graph topology and features directly from Neo4j
        graph_data = load_cora_from_neo4j(driver).to(device)

        models_to_test = [
            'Vanilla GNN (Graph Neural Network)',
            'GCN (Graph Convolutional Network)',
            'GraphSAGE (Sample and Aggregate)',
            'GAT (Graph Attention Network)',
            'GIN (Graph Isomorphism Network)'
        ]

        results = []
        hidden_dim = 64
        epochs = 200
        lr = 0.01
        weight_decay = 5e-4

        for model_name in models_to_test:
            torch.manual_seed(42)
            model = BenchmarkGNN(
                model_type=model_name,
                in_channels=graph_data.num_node_features,
                hidden_channels=hidden_dim,
                out_channels=int(graph_data.y.max().item() + 1),
                heads=8,
                dropout=0.5
            ).to(device)

            optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
            criterion = nn.NLLLoss()

            best_val_acc = 0.0
            best_test_acc = 0.0

            start_time = time.time()
            for epoch in range(1, epochs + 1):
                train_step(model, graph_data, optimizer, criterion)
                _, val_acc, test_acc = evaluate_splits(model, graph_data)

                if val_acc > best_val_acc:
                    best_val_acc = val_acc
                    best_test_acc = test_acc

            elapsed_time = time.time() - start_time
            param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)

            results.append({
                'Architecture': model_name,
                'Parameters': param_count,
                'Best Val Acc (%)': round(best_val_acc * 100, 2),
                'Test Acc @ Best Val (%)': round(best_test_acc * 100, 2),
                'Train Time (s)': round(elapsed_time, 2)
            })

        df = pd.DataFrame(results)
        print("=" * 85)
        print("         BENCHMARK RESULTS ON CORA (DATA LOADED FROM NEO4J)")
        print("=" * 85)
        print(df.to_string(index=False))

    finally:
        driver.close()


if __name__ == '__main__':
    main()